# Lab 12: Instrumental Variables — Diagnosing a Lever That Does Not Hold
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 120 min Core + 35 min Extension

---

**Learning Objectives:**
- Diagnose a weak instrument before reading its second stage, and say what the F-statistic does not tell you
- Show that with a weak instrument the inference fails first, and that with a very weak one 2SLS slides toward OLS, not toward zero
- State which population an IV estimate belongs to, and why that is not the ATE
- Argue an exclusion restriction in both directions rather than asserting it, and measure how large a violation an estimate tolerates
- Compute 2SLS by hand from two OLS regressions, and match a package to every digit

**Dataset:** Card (1995) college proximity, from the `wooldridge` package, plus simulated data where the true effect is known

**Format:** Each diagnosis hands you an estimate that is wrong or misread, and asks what happened before it tells you. Write your diagnosis before you run the reveal that follows it.

**Deliverable:** your written diagnoses (Diagnoses 1, 2 and 3), the required `iv_by_hand()` function, and `src/iv_tools.py`, a finished module you save and upload.

**Time:** Core (120 min, including the required Write it yourself) + Extension (optional, 35 min)

With setup, the README and the GitHub submission, plan on about two hours and twenty minutes, most of it after class. The optional Extension adds about 35 minutes.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Write it yourself (required, after Diagnosis 3):** an `iv_by_hand()` function, two-stage least squares from two OLS regressions plus the first-stage F, about eight lines, in an empty cell, plus one given line you copy under it. A check cell tells you when it is right. It is required.
> - **Extension (optional):** a simulation with two instruments, from a three-step outline.
>
> **You write, in text cells:** three answers in Diagnosis 1 (two before the 500 draws, one after), four in Diagnosis 2, and two in Diagnosis 3.
>
> **You run and read:** everything else. That is the weak-instrument simulation and its 500 draws, the Card (1995) estimates, the simulation with a direct effect and its sweep, and Ship It's finished module, `iv_tools.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints, and Write it yourself rebuilds Diagnosis 2's 2SLS estimate by hand.

In [ ]:
# GIVEN — run as-is
# Setup: install the two packages Colab does not have, then load the libraries
!pip install linearmodels wooldridge -q

from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import wooldridge as woo                 # the datasets of Wooldridge's textbook, Card (1995) among them
from linearmodels.iv import IV2SLS       # two-stage least squares

Path("src").mkdir(exist_ok=True)         # Ship It writes its module into this folder

RANDOM_STATE = 42    # each simulation below starts its own generator from this seed
print("Setup complete.")

## Diagnosis 1: The Instrument That Barely Moves Anything (40 min)

An analyst reports a 2SLS estimate with a large, significant second stage. They did not report
the first stage. Below is a simulation where you know the truth: the effect of `d` on `y` is
exactly 1.0, and an unobserved confounder `u` raises both, so OLS is biased upward. `pi` sets
how hard the instrument `z` pushes `d`: it is the first stage.

**Reading the code.** `np.random.default_rng(seed)` starts a random-number generator, so the
same seed gives the same data every run, and `2_000` is 2000 (the underscore only helps you
read it). `sm.add_constant(x)` adds a column of ones, the intercept, and `sm.OLS(y, X).fit()`
fits the regression; `.params` and `.tvalues` hold the coefficients and t-statistics, by name.
With one instrument, the first-stage F is the square of the instrument's t-statistic.

`IV2SLS(dependent, exog, endog, instruments)` is the `linearmodels` package's two-stage least
squares. Its four arguments come in that order: the outcome, the exogenous regressors (here
only the constant, a column of ones), the endogenous regressor, and the instrument. `.fit()`
uses robust standard errors unless told otherwise, and `.params`, `.std_errors` and `.pvalues`
hold the results. A function can return several values, as `estimates` does with
`return F, ols, iv`, and `F, ols, iv = estimates(...)` unpacks them into three names.

In [ ]:
# GIVEN — run as-is
# A simulation with a true effect of 1.0 and an instrument of strength pi
TRUE_BETA = 1.0


def simulate(pi, n=2_000, seed=RANDOM_STATE, beta=TRUE_BETA):
    """One sample of n people: instrument z, confounder u, treatment d, outcome y."""
    r = np.random.default_rng(seed)
    z = r.normal(0, 1, n)                       # the instrument
    u = r.normal(0, 1, n)                       # unobserved confounder
    d = pi * z + u + r.normal(0, 1, n)          # u raises d ...
    y = beta * d + 2.0 * u + r.normal(0, 1, n)  # ... and raises y, so OLS is biased up
    return pd.DataFrame({"y": y, "d": d, "z": z})

In [ ]:
# GIVEN — run as-is
# One sample at each strength: the first-stage F, OLS, and 2SLS with its standard error


def estimates(s):
    """The first-stage F, the OLS slope and the 2SLS fit, for one simulated sample s."""
    first = sm.OLS(s["d"], sm.add_constant(s["z"])).fit()
    ols = sm.OLS(s["y"], sm.add_constant(s["d"])).fit()
    exog = pd.DataFrame({"const": np.ones(len(s))}, index=s.index)   # the constant only
    iv = IV2SLS(s["y"], exog, s["d"], s["z"]).fit()
    return first.tvalues["z"] ** 2, ols.params["d"], iv


# {x:>10.3f} right-aligns x in 10 characters with 3 decimals
print(f"{'first stage pi':>16}{'F':>10}{'OLS':>10}{'2SLS':>10}{'SE':>8}{'p':>8}{'truth':>8}")
print("-" * 70)
for pi in [0.01, 0.05, 0.15, 0.5, 1.0]:
    F, ols, iv = estimates(simulate(pi))
    print(f"{pi:>16.2f}{F:>10.1f}{ols:>10.3f}{iv.params['d']:>10.3f}"
          f"{iv.std_errors['d']:>8.3f}{iv.pvalues['d']:>8.3f}{TRUE_BETA:>8.1f}")

**Expected output:** at pi = 1.0 the first stage is strong (F = 954.3), and 2SLS lands at 1.029
against a truth of 1.0 while OLS sits at 1.698. As pi falls, 2SLS drifts away from the truth.
At pi = 0.01 (F = 0.1) this one sample gives 3.949, with a standard error of 7.620.

### YOUR DIAGNOSIS

Write these before you run the 500 draws below. One sample cannot show where the 2SLS estimate
is heading as the instrument weakens; the reveal repeats the simulation 500 times at each
strength.

1. As the instrument weakens, does the *typical* 2SLS estimate head to the truth (1.0), to zero,
   or to the OLS estimate? Name the direction precisely, and say why: "biased" is not an answer.
2. The 2SLS standard error grows as pi falls: 0.052 at pi = 1.0, 0.851 at pi = 0.05, 7.620 at
   pi = 0.01. Predict whether that protects you from reporting a significant estimate that is
   wrong, and say why.

*Your diagnosis here:*

**The reveal: 500 draws at each strength.** The next cell runs the simulation 500 times for
each pi, each time with a new seed. It reports medians, and two columns about significance:
`% sig` is the share of draws in which 2SLS rejects beta = 0 at the 5% level, and the last
column is the share of *those* draws whose estimate is more than 0.5 away from the truth. It
takes about half a minute.

**Reading the code.** `.append(x)` adds `x` to the end of a list, and `np.array(...)` turns
the list into an array. `np.array(pvals) < 0.05` gives one True/False per draw, and the mean of
True/False values is the share of Trues. `iv_draws[sig]` keeps the draws where `sig` is True,
the significant ones, and `np.abs` is the absolute value.

In [ ]:
# GIVEN — run as-is
# The reveal: 500 samples at each strength
REPS = 500
print(f"{'pi':>6}{'med F':>9}{'med OLS':>10}{'med 2SLS':>11}{'mean 2SLS':>12}"
      f"{'% sig':>8}{'  of those, % off by >0.5':>26}")
print("-" * 82)

for pi in [0.01, 0.05, 0.15, 0.5, 1.0]:
    Fs, ols_draws, iv_draws, pvals = [], [], [], []
    for k in range(REPS):
        F, ols, iv = estimates(simulate(pi, seed=1000 + k))    # a new sample each time
        Fs.append(F)
        ols_draws.append(ols)
        iv_draws.append(iv.params["d"])
        pvals.append(iv.pvalues["d"])

    iv_draws = np.array(iv_draws)
    sig = np.array(pvals) < 0.05                               # True where 2SLS rejects beta = 0
    off_given_sig = np.mean(np.abs(iv_draws[sig] - TRUE_BETA) > 0.5)

    print(f"{pi:>6.2f}{np.median(Fs):>9.1f}{np.median(ols_draws):>10.3f}"
          f"{np.median(iv_draws):>11.3f}{np.mean(iv_draws):>12.3f}"
          f"{sig.mean() * 100:>7.1f}%{off_given_sig * 100:>25.1f}%")

In [ ]:
# GIVEN — run as-is
# The reveal, continued: one more sample, at pi = 0.03
F, ols, iv = estimates(simulate(0.03))
print(f"First-stage F : {F:>7.1f}")
print(f"OLS estimate  : {ols:>7.3f}")
print(f"2SLS estimate : {iv.params['d']:>7.3f}")
print(f"Truth         : {TRUE_BETA:>7.3f}")

**Reading the reveal.** Read the med 2SLS column from the bottom up. With a strong first stage
the median 2SLS estimate is 0.996, on the truth. At pi = 0.01 (median F = 0.5) it is **2.038**,
against a median OLS of **1.998**.

The mean 2SLS column is no use: at pi = 0.01 it is −1.525, below zero. With one instrument and
one endogenous regressor, 2SLS is a ratio whose denominator, the sample covariance of `z` and
`d`, can land arbitrarily close to zero, so its sampling distribution has no mean. That is why
the table reports medians, and why a single sample could print 3.949.

Read the median column again in the middle rows. At pi = 0.15 (median F = 22.1) the median is
0.971, and even at pi = 0.05 (median F = 2.5) it is still 1.081. With one instrument, 2SLS stays
roughly median-unbiased until the first stage is very weak, and only then slides to OLS. What
breaks first is inference (question 3).

The extra sample at pi = 0.03 gives 2SLS = 1.974, beside its OLS estimate of 2.018 and far from
the truth. On another seed it could land near 1.0 and look like a success, and the estimate
alone cannot tell you which happened. Its first-stage F of 0.9 is the number in your own output
that warns you.

### YOUR DIAGNOSIS, continued

3. **The confident wrong answer.** At pi = 0.05, 31.4% of the 500 draws reject beta = 0 at the
   5% level, and 75.2% of those significant draws are off the truth by more than 0.5. At
   pi = 0.01, 21.0% are significant, and 98.1% of those are off by more than 0.5. Explain how a
   weak instrument produces a *confident* wrong answer rather than an obviously noisy one, and
   why "my second stage was significant" is therefore no defence.

*Your diagnosis here:*

---

## Diagnosis 2: The Estimate That Belongs to Someone Else (25 min)

Card (1995) instruments schooling with growing up near a four-year college (`nearc4`). The
estimate is famously *larger* than OLS, which surprised people who expected the ability bias
to run the other way.

**Reading the code.** `woo.data("card")` loads Card's data, 3,010 young men from the National
Longitudinal Survey. The controls are experience and its square, race (`black`), living in the
South, and living in a metropolitan area (`smsa`). `["lwage", "educ", "nearc4"] + controls`
joins two lists into one, and `.dropna(subset=...)` drops any row missing one of those
columns. No row is missing one; dropping rows missing *any* column would lose 1,410 of the
3,010 men, mostly for IQ and parents' schooling, which this lab never uses. The first stage
regresses schooling on `nearc4` and the controls; as in Diagnosis 1, its F is the square of
`nearc4`'s t-statistic. In the prints, `{x:+.4f}` shows the sign even when it is +, and
`{n:,}` puts commas in a whole number.

In [ ]:
# GIVEN — run as-is
# Card (1995): the first stage, and the OLS and 2SLS returns to schooling
controls = ["exper", "exper2", "black", "south", "smsa"]
card = woo.data("card")
card["exper2"] = card["exper"] ** 2
card = card.dropna(subset=["lwage", "educ", "nearc4"] + controls)

exog = sm.add_constant(card[controls])       # the constant and the controls
card_first = sm.OLS(card["educ"], sm.add_constant(card[["nearc4"] + controls])).fit()
card_ols = sm.OLS(card["lwage"], sm.add_constant(card[["educ"] + controls])).fit()
card_iv = IV2SLS(card["lwage"], exog, card["educ"], card["nearc4"]).fit()

print(f"n = {len(card):,}")
print(f"First stage: nearc4 coefficient {card_first.params['nearc4']:+.4f}, "
      f"t = {card_first.tvalues['nearc4']:.2f}, F = {card_first.tvalues['nearc4'] ** 2:.1f}")
print()
print(f"OLS  return to schooling : {card_ols.params['educ']:.4f}")
print(f"2SLS return to schooling : {card_iv.params['educ']:.4f}")

### YOUR DIAGNOSIS

1. The IV estimate exceeds OLS. If the only problem were ability bias, IV should have come in *below* OLS. Give two explanations consistent with what you see.
2. Name the population this estimate belongs to. Who are the compliers here, and are they the group a policymaker cares about?
3. State the exclusion restriction for proximity to college in one sentence. Then give the strongest argument against it: something proximity could affect other than schooling.
4. The first-stage F is 16.7, above the rule of thumb of 10. What does the F measure, and why is passing that rule not enough?

*Your diagnosis here:*

---

## Diagnosis 3: When Exclusion Is Only Nearly True (15 min)

Exclusion cannot be tested. You can, however, ask how large a violation your conclusion
tolerates. Below, the instrument has a small direct effect on the outcome, `gamma`: the arrow
that the exclusion restriction says is absent. With `gamma = 0` exclusion holds. Everything
else is Diagnosis 1's simulation, with a strong instrument (pi = 0.5) and 5,000 people. The
second cell sweeps `gamma` from 0 to 0.5 and prints the 2SLS estimate at each value.

**Reading the code.** `np.linspace(0, 0.5, 11)` gives the eleven values 0, 0.05, ..., 0.5.
`estimates()` returns three things; `_, _, iv = estimates(...)` keeps only the third, and `_`
is the usual name for a value you do not need.

In [ ]:
# GIVEN — run as-is
# Diagnosis 1's simulation, plus a direct path from z to y of size gamma


def simulate_violation(gamma, pi=0.5, n=5_000, seed=RANDOM_STATE, beta=TRUE_BETA):
    """Like simulate(), but y also depends on z directly: gamma = 0 means exclusion holds."""
    r = np.random.default_rng(seed)
    z = r.normal(0, 1, n)
    u = r.normal(0, 1, n)
    d = pi * z + u + r.normal(0, 1, n)
    y = beta * d + gamma * z + 2.0 * u + r.normal(0, 1, n)
    return pd.DataFrame({"y": y, "d": d, "z": z})

In [ ]:
# GIVEN — run as-is
# The sweep: the 2SLS estimate at each gamma from 0 to 0.5
for gamma in np.linspace(0, 0.5, 11):
    _, _, iv = estimates(simulate_violation(gamma))
    print(f"gamma = {gamma:.2f}   2SLS = {iv.params['d']:.3f}")

At `gamma = 0` the estimate is 0.838, not exactly 1.0: one sample of 5,000 people still has
sampling error.

### YOUR DIAGNOSIS

1. Each 0.05 of `gamma` adds the same amount to the estimate. Find it, divide 0.05 by it, and say which coefficient of the simulation it matches.
2. Measured from the estimate at `gamma = 0` (0.838), at which `gamma` on the grid has the estimate moved by more than 0.5, half the true effect? Write that as the sentence a referee needs: "a direct effect of the instrument larger than X would move our estimate by more than Y". Then say whether you find it reassuring here, and why.

*Your answers here:*

1.

2.

---

## Write it yourself (required — 30 min)

This is the one function in this lab you write from an empty cell, and it is required. No AI:
write it yourself.

Diagnosis 2 let the `linearmodels` package compute Card's 2SLS estimate. Two-stage least
squares is two OLS regressions, as in class:
- **First stage:** regress the endogenous variable on the instrument and the controls, with a constant.
- **Second stage:** regress the outcome on the first stage's fitted values and the same controls, with a constant. The coefficient on the fitted values is the 2SLS estimate.

With one instrument, the reduced form's coefficient on the instrument divided by the first
stage's gives the same number, and you may compute it that way instead.

Don't report the second stage's `sm.OLS` standard error: its residuals use the fitted values,
not `educ`, so it is too small. That is why `iv_by_hand` returns only the estimate and the F.

In the empty cell below:
1. Write a function `iv_by_hand(df, outcome, endog, instrument, controls)` that returns two
   numbers, in this order: the 2SLS estimate, and the first-stage F for the instrument, computed
   as Diagnosis 2 computed it. `outcome`, `endog` and `instrument` are column names, and
   `controls` is a list of column names. Use `sm.OLS`, not `IV2SLS`.
2. Under the function, copy this line unchanged. It runs your function on the specification
   from class, with only `exper` and `exper2` as controls:

   `short_beta, short_F = iv_by_hand(card, "lwage", "educ", "nearc4", ["exper", "exper2"])`

The function takes about eight lines. A fitted regression's `.fittedvalues` holds its fitted
values, one per row. If you display the pair your function returns, it may show as
`(np.float64(0.13...), np.float64(16.7...))`: that is numpy's way of writing a number, and it is
fine. The check runs your function on Diagnosis 2's specification, where it should reproduce
`linearmodels` to every digit (0.1323 and F = 16.7), and then with other column names, other
controls and another sample, where each common slip gives a different answer. Until you write
the cell, the check says "Not written yet"; once you do, it fails with a message until your
function is right.

The deck's Card slides used the shorter specification: 2SLS = 25.9% and F = 59.07. That F is
the robust version, which `linearmodels` reports by default, so your classical F comes out a
little lower.

In [ ]:
# WRITE IT YOURSELF — required: iv_by_hand, then the short_beta, short_F line from the task

In [ ]:
# CHECK — for iv_by_hand above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "iv_by_hand" not in globals():
    print("Not written yet: define iv_by_hand in the cell above, then run this check again.")
else:
    import inspect

    def package(df, ctrl, outcome="lwage", endog="educ", instrument="nearc4"):
        """linearmodels' 2SLS estimate, and its classical first-stage F."""
        fit = IV2SLS(df[outcome], sm.add_constant(df[ctrl]), df[endog], df[instrument]).fit(
            cov_type="unadjusted", debiased=True)
        return fit.params[endog], fit.first_stage.diagnostics["f.stat"].iloc[0]

    def same(a, b):
        """Equal to about eight significant digits: only rounding inside the computer passes."""
        return bool(np.isclose(a, b, rtol=1e-8, atol=0))

    def rounded(a, b):
        """True if a is b rounded to some number of decimals, but not b itself."""
        return not same(a, b) and any(a == round(b, k) for k in range(7))

    ARGUMENT_OF = {"lwage": "outcome", "educ": "endog", "nearc4": "instrument"}

    def run(where, df, outcome, endog, instrument, ctrl, on_error=""):
        """Call iv_by_hand; return its two results as numbers, or stop with a plain message."""
        try:
            got = iv_by_hand(df, outcome, endog, instrument, ctrl)
        except KeyError as err:
            name = err.args[0] if err.args else ""
            if name in ARGUMENT_OF:
                hint = f" Does it use its {ARGUMENT_OF[name]} argument, or '{name}'?"
            elif name in (outcome, endog, instrument):
                hint = " A regression in it lacks that column: does every stage use df and the arguments?"
            else:
                hint = on_error
            raise AssertionError(f"{where}, iv_by_hand stopped with KeyError {err}.{hint}") from None
        except Exception as err:
            raise AssertionError(f"{where}, iv_by_hand stopped with {type(err).__name__}: "
                                 f"{err}.{on_error}") from None
        assert got is not None, "iv_by_hand gives back nothing: use return, not print."
        assert isinstance(got, (tuple, list, np.ndarray)) and len(got) == 2, (
            f"iv_by_hand should return two numbers, as return estimate, F. It returned a "
            f"{type(got).__name__} of length {len(got) if hasattr(got, '__len__') else 1}.")
        numbers = []
        for label, value in zip(["estimate", "F"], got):
            try:
                numbers.append(float(value))
            except (TypeError, ValueError):
                raise AssertionError(
                    f"iv_by_hand's {label} is a {type(value).__name__}, not a number. Indexing "
                    "with a list, as in [[name]], gives a table; [name] gives a number.") from None
        return numbers

    assert len(card) == 3010 and controls == ["exper", "exper2", "black", "south", "smsa"], (
        "card or controls is no longer Diagnosis 2's. Give the names in your cell other names, "
        "then rerun the notebook from the top.")
    # inspect.signature lists the names of a function's arguments, in order
    params = list(inspect.signature(iv_by_hand).parameters)
    assert params[:5] == ["df", "outcome", "endog", "instrument", "controls"], (
        f"iv_by_hand's arguments are ({', '.join(params)}). The check calls it as the task asks: "
        "iv_by_hand(df, outcome, endog, instrument, controls).")
    try:
        source = inspect.getsource(iv_by_hand)
    except (OSError, TypeError):
        source = ""
    assert "IV2SLS" not in source and "linearmodels" not in source, (
        "iv_by_hand uses linearmodels. Write it with two OLS regressions.")

    passed = list(controls)        # a copy, to see whether iv_by_hand changes the list it is given
    beta, F = run("On Diagnosis 2's specification", card.copy(), "lwage", "educ", "nearc4", passed)
    assert passed == controls, (
        f"iv_by_hand changed the controls list it was given; it is now {passed}. Build a new "
        "list of regressors rather than adding to controls.")
    want_beta, want_F = package(card, controls)

    # Diagnosis 2's specification: the estimate
    assert not (same(beta, want_F) and same(F, want_beta)), (
        "iv_by_hand returned the F first. Return the 2SLS estimate first, then the F.")
    slips = {
        7.559217: "is upside down: the first stage's coefficient divided by the reduced form's",
        0.074009: "is Diagnosis 2's OLS estimate: the second stage used educ itself, not the "
                  "first stage's fitted values",
        0.188063: "is 2SLS with no controls at all, the Wald estimate from class. Both stages "
                  "need the controls",
        0.053827: "comes from a first stage without the controls. The first stage needs the "
                  "controls as well as the instrument",
        0.028180: "comes from a second stage without the controls. It needs the same controls "
                  "as the first stage",
        0.020385: "comes from a first stage without a constant: sm.OLS adds none by itself",
        0.354877: "comes from a second stage without a constant: sm.OLS adds none by itself",
        0.322844: "comes from two regressions without a constant: sm.OLS adds none by itself",
        0.065285: "comes from 1,600 men, not 3,010: every row missing any column was dropped. "
                  "Use df as it is given",
    }
    for value, cause in slips.items():
        assert abs(beta - value) > 1e-5, f"iv_by_hand gives {beta:.4f} on Diagnosis 2's specification. That {cause}."
    assert not rounded(beta, want_beta), (
        f"iv_by_hand gives {beta}, and linearmodels {float(want_beta)}. Is the estimate "
        "rounded? Return it unrounded.")
    assert same(beta, want_beta), (
        f"On Diagnosis 2's specification iv_by_hand gives {beta:.6f}; linearmodels gives "
        f"{want_beta:.6f}. Compare each stage with the recipe above.")

    # Diagnosis 2's specification: the first-stage F
    F_slips = {
        451.8655: "is the F of the whole first-stage regression, which tests the controls too. "
                  "The F for the instrument tests nearc4 alone",
        115.1636: "is the F of the whole second-stage regression. The F you need is the first "
                  "stage's, for the instrument",
        0.0504: "is the second stage's standard error, not an F",
        4.0887: "is nearc4's t-statistic. With one instrument the F is its square",
        17.5133: "is a robust F. Use the classical one, as Diagnosis 2 printed",
        17.5541: "is a robust F. Use the classical one, as Diagnosis 2 printed",
        17.4517: "is a robust (HC3) F. Use the classical one, as Diagnosis 2 printed",
        6.8811: "comes from a regression of the outcome (the reduced form or the second stage). "
                "The first-stage F comes from the regression of educ",
        63.9119: "comes from a first stage without the controls",
    }
    for value, cause in F_slips.items():
        assert abs(F - value) > 1e-4 * max(1, value), f"iv_by_hand's F is {F:.4f} on Diagnosis 2's specification. That {cause}."
    assert not rounded(F, want_F), (
        f"iv_by_hand's F is {F}, and linearmodels' {float(want_F)}. Is the F rounded? "
        "Return it unrounded.")
    assert same(F, want_F), (
        f"On Diagnosis 2's specification iv_by_hand's F is {F:.4f}; Diagnosis 2 printed "
        f"{want_F:.1f}.")

    # Does the function use each argument? Another instrument, other column names, other
    # controls, another sample
    near2 = run("With instrument='nearc2'", card.copy(), "lwage", "educ", "nearc2", list(controls))
    assert not (same(near2[0], want_beta) or same(near2[1], want_F)), (
        "With instrument='nearc2', iv_by_hand gives nearc4's answer. Does it use its "
        "instrument argument, or 'nearc4'?")
    want_near2 = package(card, controls, instrument="nearc2")
    assert same(near2[0], want_near2[0]) and same(near2[1], want_near2[1]), (
        f"With instrument='nearc2', iv_by_hand gives ({near2[0]:.6f}, {near2[1]:.4f}); "
        f"linearmodels gives ({want_near2[0]:.6f}, {want_near2[1]:.4f}).")
    north = card[card["south"] == 0].copy()         # the men outside the South
    north_controls = ["exper", "exper2", "black", "smsa"]
    north_got = run("On the men outside the South", north, "lwage", "educ", "nearc4",
                    list(north_controls), on_error=" Does every stage use df, or card?")
    assert not same(north_got[0], package(card, north_controls)[0]), (
        "On a sample of the men outside the South, iv_by_hand gives the answer for all 3,010 men. "
        "Does it use its df argument, or card?")
    want_north = package(north, north_controls)
    assert same(north_got[0], want_north[0]) and same(north_got[1], want_north[1]), (
        f"On the men outside the South, iv_by_hand gives ({north_got[0]:.6f}, "
        f"{north_got[1]:.4f}); linearmodels gives ({want_north[0]:.6f}, {want_north[1]:.4f}).")

    renamed = card.rename(columns={"lwage": "log_wage", "educ": "schooling", "nearc4": "near_college"})
    named = run("With the columns renamed", renamed, "log_wage", "schooling", "near_college",
                list(controls))
    assert same(named[0], want_beta) and same(named[1], want_F), (
        f"With the columns renamed (log_wage, schooling, near_college), iv_by_hand gives "
        f"({named[0]:.6f}, {named[1]:.4f}) instead of ({want_beta:.6f}, {want_F:.4f}). Does it "
        "use its outcome, endog and instrument arguments everywhere?")
    short = ["exper", "exper2"]
    short_got = run("With controls exper and exper2 only", card.copy(), "lwage", "educ", "nearc4",
                    list(short))
    assert not same(short_got[0], want_beta), (
        "With controls exper and exper2 only, iv_by_hand gives the same estimate as with "
        "Diagnosis 2's five controls. Does it use its controls argument, or the list named "
        "controls from Diagnosis 2?")
    want_short = package(card, short)
    assert same(short_got[0], want_short[0]) and same(short_got[1], want_short[1]), (
        f"With controls exper and exper2 only, iv_by_hand gives ({short_got[0]:.6f}, "
        f"{short_got[1]:.4f}); linearmodels gives ({want_short[0]:.6f}, {want_short[1]:.4f}).")
    # The given line
    assert "short_beta" in globals() and "short_F" in globals(), (
        "iv_by_hand works. Now copy the short_beta, short_F line from the task under it.")
    assert same(float(short_beta), short_got[0]) and same(float(short_F), short_got[1]), (
        f"short_beta and short_F ({float(short_beta):.4f}, {float(short_F):.1f}) are not "
        "iv_by_hand with only exper and exper2 as controls. Copy the line from the task unchanged.")
    print(f"iv_by_hand matches linearmodels on Diagnosis 2's specification: {beta:.4f}, F = {F:.1f}.")
    print(f"With only exper and exper2 as controls: 2SLS = {float(short_beta):.4f}, F = {float(short_F):.1f}.")

---

## Ship It: `src/iv_tools.py` (10 min)

Here is the finished module. Its design rule is the one from class: never report a second stage
without the first-stage diagnostics beside it, so `iv_estimate` returns the first-stage F with
every estimate. `exclusion_sensitivity` asks Diagnosis 3's question of real data. If the
instrument had a direct effect `gamma` on the outcome, subtracting `gamma` times the instrument
from the outcome removes that path, and 2SLS on what is left is the estimate you would then
have got.

This part is reading only: run the two cells and compare the output with the expected output
below. The first saves the module as a file, the second imports it and runs it on Card.

**Reading the cells.** `%%writefile src/iv_tools.py` saves the cell as that file instead of
running it. The notes after a colon or `->` (`instruments: list[str]`, `-> dict`) only say what
type each argument and result is, and `list[str] | None` means a list of names, or `None`.
`controls or []` is `controls` if one was given and an empty list if it is `None`.
`[f"{z} = 0" for z in instruments]` builds one string per instrument, such as `"nearc4 = 0"`,
and `.f_test(...)` tests that all those coefficients are zero; with one instrument its F is the
square of the t-statistic. In the second cell, `.round(4)` rounds every number in the table to
four decimals, and `.to_string(index=False)` prints it without row numbers.

In [ ]:
%%writefile src/iv_tools.py
"""Instrumental-variables helpers that never report a second stage alone (ECON 5200 Lab 12).

The design rule: every 2SLS estimate comes back with its first-stage F beside it,
because an IV estimate read without its first stage cannot be interpreted.
"""
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS


def first_stage(df: pd.DataFrame, endog: str, instruments: list[str],
                controls: list[str] | None = None) -> dict:
    """The first-stage coefficients on the instruments, their joint F, and n."""
    controls = controls or []
    fit = sm.OLS(df[endog], sm.add_constant(df[instruments + controls])).fit()
    hypotheses = [f"{z} = 0" for z in instruments]    # every instrument's coefficient is zero
    test = fit.f_test(hypotheses)
    return {"coef": fit.params[instruments], "F": float(test.fvalue), "n": int(fit.nobs)}


def iv_estimate(df: pd.DataFrame, outcome: str, endog: str, instruments: list[str],
                controls: list[str] | None = None) -> pd.Series:
    """2SLS, with robust standard errors, and the first-stage F attached."""
    controls = controls or []
    fit = IV2SLS(df[outcome], sm.add_constant(df[controls]), df[endog], df[instruments]).fit()
    return pd.Series({"estimate": fit.params[endog],
                      "std_err": fit.std_errors[endog],
                      "first_stage_F": first_stage(df, endog, instruments, controls)["F"],
                      "n": fit.nobs})


def exclusion_sensitivity(df: pd.DataFrame, outcome: str, endog: str, instrument: str,
                          gammas, controls: list[str] | None = None) -> pd.DataFrame:
    """The 2SLS estimate if the instrument had a direct effect gamma on the outcome.

    Subtracting gamma * instrument from the outcome removes that direct path: Conley,
    Hansen and Rossi's (2012) "plausibly exogenous" idea, one gamma at a time.
    """
    rows = []
    for gamma in gammas:
        adjusted = df.copy()
        adjusted[outcome] = df[outcome] - gamma * df[instrument]
        estimate = iv_estimate(adjusted, outcome, endog, [instrument], controls)["estimate"]
        rows.append({"gamma": gamma, "estimate": estimate})
    return pd.DataFrame(rows)

In [ ]:
# GIVEN — run as-is
# Import the module and run it on Card: the estimate and F should match Diagnosis 2
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import iv_tools
importlib.reload(iv_tools)         # reload, so a re-run of the %%writefile cell takes effect

result = iv_tools.iv_estimate(card, "lwage", "educ", ["nearc4"], controls)
print(f"2SLS {result['estimate']:.4f}  (std err {result['std_err']:.4f}), "
      f"first-stage F {result['first_stage_F']:.1f}, n = {result['n']:,.0f}")

sensitivity = iv_tools.exclusion_sensitivity(card, "lwage", "educ", "nearc4",
                                             [0, 0.01, 0.02, 0.03], controls)
print()
print(sensitivity.round(4).to_string(index=False))

**Expected output:** 2SLS 0.1323 (std err 0.0485), first-stage F 16.7 and n = 3,010, as in
Diagnosis 2. Then the sensitivity table: 0.1323 at `gamma = 0`, 0.1026 at 0.01, 0.0730 at 0.02
and 0.0434 at 0.03. A direct effect of 0.02, growing up near a college raising wages by about 2%
for some reason other than schooling, is enough to take the 2SLS estimate below OLS's 0.0740.

---

## Extension (Optional — 35 min): LATE and Who It Belongs To

Angrist and Imbens showed that IV recovers a local average treatment effect: the effect for the
compliers, the people whose treatment the instrument changes. The instrument defines who the
compliers are, so two valid instruments for the same treatment can estimate two different
effects.

**The task.** In the cell below, and in new cells under it:
1. Simulate 10,000 people with two independent coin-flip instruments, `z1` and `z2`. Group A
   (40% of people) takes the treatment exactly when `z1` is 1, group B (40%) exactly when `z2`
   is 1, and the other 20% always take it and also have a higher outcome without it. The
   treatment raises the outcome by 2.0 in group A, by 0.5 in group B and by 1.0 for the rest.
2. Estimate 2SLS with `z1` as the only instrument, then with `z2`, using `IV2SLS` as in
   Diagnosis 1. Print OLS beside them.
3. Answer the questions below.

In [ ]:
# YOUR TASK — optional: two valid instruments, two different estimates

### Interpretation Questions

1. Both of your estimates are consistent. Consistent for what? Say what each one estimates in this simulation.
2. A specification test that rejects the equality of the two estimates is not evidence that one instrument is invalid. Why not?
3. A referee asks which of the two is "the" effect. What would you report?

*Your answers here:*

1.

2.

3.

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Simulated a weak instrument where the true effect was 1.0 and OLS was
  biased upward: at a median first-stage F of [YOUR VALUE], the median 2SLS
  estimate was [YOUR VALUE], against a median OLS estimate of [YOUR VALUE],
  and a significant 2SLS estimate was usually far from the truth
* Replicated Card's (1995) college-proximity instrument for schooling:
  first-stage F [YOUR VALUE], OLS return [YOUR VALUE], 2SLS return
  [YOUR VALUE]
* Measured how far a small direct effect of the instrument on the outcome
  moves the 2SLS estimate
* Wrote my own two-stage least squares function from two OLS regressions,
  and checked it against the linearmodels package

**Please write a README.md entry including:**
1. Project Title: Instrumental Variables: Weak Instruments, Compliers and the Exclusion Restriction
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust',
'comprehensive' or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab12-instrumental-variables`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab12-instrumental-variables`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab12-instrumental-variables`, branch `main`,
   commit message `Lab 12 submission`. This pushes the notebook, and the
   notebook's rendered output is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/iv_tools.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.